# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets and their fields
print("Available record sets and their @id:")
for record_set in metadata.record_sets:
    print(f"- RecordSet name: {record_set.name} | @id: {record_set.id}")
    print("  Fields in this RecordSet:")
    for field in record_set.fields:
        print(f"    - Field name: {field.name} | @id: {field.id} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose record set @id to extract data
# Please refer to the printed record set @ids in the previous cell.
# For this example, we'll select the first record set found.
# Replace 'record_set_id' with the actual @id for your desired record set.
record_sets = [record_set.id for record_set in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet '@id': {record_set_id} | Shape: {df.shape}")

# For demonstration, let's examine the columns of the first record set
if record_sets:
    first_rsid = record_sets[0]
    print(dataframes[first_rsid].columns.tolist())
    display(dataframes[first_rsid].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
# Select numeric fields for analysis from the first record set
first_record_set_id = record_sets[0]
df = dataframes[first_record_set_id]

# Detect numeric columns from field metadata or dataframe automatically
# Here we'll look for likely numeric columns by dtype
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields in '{first_record_set_id}' RecordSet:", numeric_fields)

if numeric_fields:
    # Let's choose the first numeric field detected
    numeric_field = numeric_fields[0]

    # Filtering example: keep records where value > threshold (e.g., 10)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records in '{first_record_set_id}' with {numeric_field} > {threshold}: {filtered_df.shape[0]} rows")
    display(filtered_df.head())

    # Normalize this numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"First few normalized values for '{numeric_field}':")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Optional: group by a categorical field if present
    group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()  # Mean by group
        print(f"Mean '{numeric_field}' by '{group_field}':")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of the selected numeric field
if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# If a group field is available, show boxplot
if numeric_fields and group_fields:
    plt.figure(figsize=(16, 5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded metadata and explored record sets and field definitions using their `@id` entries.
- Extracted and displayed data from the first available record set.
- Performed filtering, normalization, and grouping on a numeric field referenced by its `@id`.
- Visualized the distribution and group-wise differences for key fields.

> Next steps: You may extend this notebook by selecting specific record sets/fields, conducting more advanced filtering, or creating additional visualizations relevant to the biological or experimental context of this dataset.